# Boltz-2 Biomolecular Structure & Binding Affinity Prediction with `foldrun-shim`

This notebook adapts the official [**`jwohlwend/boltz`**](https://github.com/jwohlwend/boltz) examples (`affinity.yaml`, `prot.yaml`, `pocket.yaml`, `prot_custom_msa.yaml`, `prot_no_msa.yaml`) and public Boltz-2 Colab workflows to execute seamlessly on **FoldRun**'s auto-scaling Google Cloud Vertex AI A100 Pipelines using **`foldrun-shim`**.


---

## Original Notebook Reference, Credits & Citations

- **Original Public Examples & Notebooks**:
  - Official [`jwohlwend/boltz` Examples](https://github.com/jwohlwend/boltz/tree/main/examples) ([`affinity.yaml`](https://github.com/jwohlwend/boltz/blob/main/examples/affinity.yaml), [`prot.yaml`](https://github.com/jwohlwend/boltz/blob/main/examples/prot.yaml), [`pocket.yaml`](https://github.com/jwohlwend/boltz/blob/main/examples/pocket.yaml), [`prot_custom_msa.yaml`](https://github.com/jwohlwend/boltz/blob/main/examples/prot_custom_msa.yaml), [`prot_no_msa.yaml`](https://github.com/jwohlwend/boltz/blob/main/examples/prot_no_msa.yaml)) & [`docs/prediction.md`](https://github.com/jwohlwend/boltz/blob/main/docs/prediction.md)
  - Public Colab Notebook: [`sokrypton/ColabFold` `Boltz1.ipynb`](https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/Boltz1.ipynb)
- **Upstream Repository**: [`jwohlwend/boltz`](https://github.com/jwohlwend/boltz)
- **Authors & Credits**: Saro Passaro, Gabriele Corso, Jeremy Wohlwend, Mateo Reveiz, Stephan Thaler, Vignesh Ram Somnath, Noah Getz, Tally Portnoi, Julien Roy, Hannes Stark, David Kwabi-Addo, Dominique Beaini, Tommi Jaakkola, and Regina Barzilay (**MIT Jameel Clinic / CSAIL & Recursion**); ColabFold notebook contributions by Sergey Ovchinnikov and Martin Steinegger.
- **License**: MIT License.
- **Primary Citations**:
  1. Passaro, S., Corso, G., Wohlwend, J. *et al.* "Boltz-2: Towards Accurate and Efficient Binding Affinity Prediction." *bioRxiv* (2025). [doi:10.1101/2025.06.14.659707](https://doi.org/10.1101/2025.06.14.659707)
  2. Wohlwend, J., Corso, G., Passaro, S. *et al.* "Boltz-1: Democratizing Biomolecular Interaction Modeling." *bioRxiv* (2024). [doi:10.1101/2024.11.19.624167](https://doi.org/10.1101/2024.11.19.624167)

---

## Why Run Boltz-2 Through `foldrun-shim` Instead of a Standard Public Colab Notebook?

| Capability | Public Boltz-2 Colab Notebook (`!boltz predict ... --use_msa_server`) | FoldRun Cloud Execution via `foldrun-shim` |
| :--- | :--- | :--- |
| **MSA Generation & Data Privacy** | Sends sequences over the public internet to `https://api.colabfold.com` (`--use_msa_server`), exposing proprietary sequences and hitting strict IP rate limits / queue throttling. | Runs CPU-optimized `jackhmmer` (`--cpu 8`) on dedicated `c2-standard-16` VMs inside your private VPC against pre-provisioned **UniRef90** and **MGnify** on a 2.5 TB `BASIC_SSD` Cloud Filestore NFS share (`/mnt/nfs/foldrun`). |
| **MSA Caching Across Runs** | Re-queries the remote MMseqs2 server whenever the Colab runtime resets. | Automatically caches alignments by sequence SHA-256 at `/mnt/nfs/foldrun/boltz2_msas_cache/protein_<sha256>/combined.a3m`. Subsequent complexes sharing a receptor chain skip `jackhmmer` completely (`0s` MSA latency). |
| **Model Weights & CCD Ligands** | Downloads multi-GB `boltz2_conf.ckpt`, `boltz2_aff.ckpt`, and `mols.tar` to `~/.boltz` on every fresh Colab session. | Pre-staged on SSD Filestore at `/mnt/nfs/foldrun/boltz2/cache` (`boltz2_conf.ckpt`, `boltz2_aff.ckpt`, and extracted `mols/` Chemical Component Dictionary) and mounted directly into every worker container. |
| **GPU Hardware & Multi-Seed Scaling** | Single Colab T4 GPU (16 GB VRAM) frequently OOMs on multimers, `--use_potentials`, or binding affinity ensembles, and runs seeds sequentially. | Fans out `--num_model_seeds` (`1..10`) in parallel across dedicated NVIDIA A100 GPUs via Vertex AI `dsl.ParallelFor`, auto-selecting **A100 40 GB** (`a2-highgpu-1g`) for $\le 2000$ tokens or **A100 80 GB** (`a2-ultragpu-1g`) for $> 2000$ tokens with DWS `FLEX_START`. |
| **Local Custom MSAs & Templates** | Requires manual file uploads to ephemeral Colab disk. | `foldrun-shim` automatically inspects `sequences[*].protein.msa` and `templates[*].cif`/`pdb` in your local YAML, bundles them into `assets.tar.gz`, stages them to `/gcs/.../inputs/assets/`, and bypasses `jackhmmer`. |
| **Notebook Disconnect Resilience** | Browser tab closure or Wi-Fi drop kills the GPU process and loses all progress. | Persists `<out_dir>/.foldrun_job.json` upon `202 Accepted` submission. Re-running an interrupted cell automatically re-attaches to the running Vertex AI `PipelineJob` and downloads the native `boltz_results_<stem>/` directory when finished. |


## 1. Prerequisites, Authentication & Installing `foldrun-shim` from GCS

During FoldRun deployment (`./deploy-all.sh` or `./deploy-all.sh --steps build --build-target sdk`), Cloud Build packages `foldrun-shim` into a pure-Python wheel and uploads both the wheel and all 7 modified public notebooks to your project's FoldRun GCS bucket under `gs://<PROJECT_ID>-foldrun-data/sdk/`:
- **Pre-Built Wheel**: `gs://<PROJECT_ID>-foldrun-data/sdk/latest_build` (and dated incremental builds)
- **Modified Notebooks**: `gs://<PROJECT_ID>-foldrun-data/sdk/notebooks/<model>/*.ipynb`

### Step-by-Step Setup (Executed in the Next Cell)

1. **Step 1 — Authenticate to Google Cloud First**:
   - **Google Colab**: Calls `from google.colab import auth; auth.authenticate_user()` so `gcloud storage cp` and `gcloud auth print-identity-token` have valid credentials before downloading the wheel.
   - **Colab Enterprise / Vertex AI Workbench**: Authenticated automatically via the runtime's attached service account and the Google Cloud Metadata Server.
   - **Local Jupyter Kernel**: Authenticate beforehand with `gcloud auth login`.
2. **Step 2 — Automatic Configuration Loading (`.env`)**:
   - FoldRun's deployment script (`./deploy-all.sh`) generates local `.env` files in `sdk/foldrun-shim/notebooks/` and uploads `gs://${FOLDRUN_GCS_BUCKET}/sdk/.env`.
   - The cell below automatically loads `.env` (searching `./.env`, `../.env`, `/content/.env`, or downloading from `gs://${FOLDRUN_GCS_BUCKET}/sdk/.env`), populating `GCP_PROJECT_ID`, `GCP_REGION`, `FOLDRUN_GCS_BUCKET`, `FOLDRUN_GATEWAY_URL`, and `FOLDRUN_VIEWER_URL`.
3. **Step 3 — Download & Install `foldrun-shim` from the FoldRun GCS Bucket**:
   Requires `roles/storage.objectViewer` on `gs://<PROJECT_ID>-foldrun-data` (ideally scoped to the `sdk/` prefix via an IAM condition):
   ```bash
   gcloud storage cp gs://${FOLDRUN_GCS_BUCKET}/sdk/latest_build /tmp/
   WHL=$(cat /tmp/latest_build)
   gcloud storage cp gs://${FOLDRUN_GCS_BUCKET}/sdk/${WHL} /tmp/
   pip install -q /tmp/${WHL}
   ```
4. **Step 4 — Configure `FOLDRUN_GATEWAY_URL` & OIDC Gateway Auth (`roles/run.invoker`)**:
   `foldrun_shim.auth.fetch_gateway_headers()` automatically attaches `Authorization: Bearer <OIDC_ID_TOKEN>` to every gateway request via:
   1. `FOLDRUN_ID_TOKEN` (if explicitly set),
   2. `FOLDRUN_IMPERSONATE_SERVICE_ACCOUNT` (`gcloud auth print-identity-token --impersonate-service-account=...`),
   3. Google Cloud Metadata Server (`google.oauth2.id_token.fetch_id_token` on Vertex AI Workbench / Colab Enterprise),
   4. `gcloud auth print-identity-token` fallback (Colab / local session).

In [ ]:
# @title 1. Authenticate to Google Cloud & Install `foldrun-shim` from GCS
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import sysconfig
from pathlib import Path


def _ensure_scripts_on_path() -> None:
    """Ensure pip console scripts (including --user installs) are discoverable on PATH."""
    script_dirs: list[str] = []
    for scheme in (None, f"{os.name}_user"):
        try:
            p = (
                sysconfig.get_path("scripts")
                if scheme is None
                else sysconfig.get_path("scripts", scheme)
            )
            if p:
                script_dirs.append(p)
        except Exception:
            pass
    script_dirs.append(str(Path(sys.executable).parent))
    current_parts = os.environ.get("PATH", "").split(os.pathsep) if os.environ.get("PATH") else []
    for d in reversed(script_dirs):
        if d and d not in current_parts:
            current_parts.insert(0, d)
    os.environ["PATH"] = os.pathsep.join(current_parts)


def _is_placeholder_value(val: str) -> bool:
    lowered = (val or "").strip().lower()
    return not lowered or any(
        tok in lowered
        for tok in (
            "your-project-foldrun-data",
            "your-project",
            "your_project",
            "example",
            "<",
        )
    )


def _is_placeholder_bucket(bucket: str) -> bool:
    clean = (bucket or "").strip().removeprefix("gs://").rstrip("/")
    return _is_placeholder_value(clean)


_ensure_scripts_on_path()


def _load_foldrun_dotenv(custom_path: Path | None = None) -> Path | None:
    """Load key=value pairs from a local .env file generated by deploy-all.sh."""
    if custom_path is not None:
        candidates = [custom_path]
    else:
        candidates = []
        if os.environ.get("FOLDRUN_ENV_FILE"):
            candidates.append(Path(os.environ["FOLDRUN_ENV_FILE"]))
        candidates.extend(
            [Path(".env"), Path("../.env"), Path("../../.env"), Path("/content/.env")]
        )
    loaded_from: Path | None = None
    for path in candidates:
        if not path.is_file():
            continue
        for raw_line in path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            k = re.sub(r"^export(?=\s)\s*", "", k.strip())
            v = v.strip().strip("'").strip('"')
            if k and v:
                existing = os.environ.get(k, "")
                if not existing or _is_placeholder_value(existing):
                    os.environ[k] = v
        if loaded_from is None:
            loaded_from = path.resolve()
        if custom_path is not None or not _is_placeholder_value(
            os.environ.get("FOLDRUN_GATEWAY_URL", "")
        ):
            return path.resolve()
    return loaded_from


_dotenv_path = _load_foldrun_dotenv()
if _dotenv_path:
    print(f"Loaded FoldRun configuration from {_dotenv_path}")

os.environ.setdefault("FOLDRUN_GCS_BUCKET", "your-project-foldrun-data")
os.environ.setdefault("FOLDRUN_GATEWAY_URL", "https://foldrun-gateway-example-uc.a.run.app")
# Optional: set FOLDRUN_IMPERSONATE_SERVICE_ACCOUNT if your user account delegates to an invoker SA
# os.environ["FOLDRUN_IMPERSONATE_SERVICE_ACCOUNT"] = "foldrun-invoker@your-project.iam.gserviceaccount.com"

# 1. Authenticate to Google Cloud first (required in standard Google Colab before accessing GCS)
if "google.colab" in sys.modules:
    from google.colab import auth as colab_auth

    colab_auth.authenticate_user()


def _install_foldrun_shim_from_gcs(gcs_bucket: str) -> None:
    """Download latest foldrun_shim wheel via latest_build from gs://<bucket>/sdk/ and install via pip."""
    clean_bucket = (gcs_bucket or "").strip().removeprefix("gs://").rstrip("/")
    bucket_configured = not _is_placeholder_bucket(clean_bucket)

    if bucket_configured and _is_placeholder_value(os.environ.get("FOLDRUN_GATEWAY_URL", "")):
        try:
            env_dst = Path("/tmp/foldrun_sdk.env")
            res = subprocess.run(
                [
                    "gcloud",
                    "storage",
                    "cp",
                    f"gs://{clean_bucket}/sdk/.env",
                    str(env_dst),
                ],
                check=False,
                capture_output=True,
                text=True,
            )
            if res.returncode == 0 and env_dst.is_file():
                if _load_foldrun_dotenv(env_dst):
                    print(f"Loaded remote FoldRun configuration from gs://{clean_bucket}/sdk/.env")
        except Exception as exc:
            print(f"Notice: could not load gs://{clean_bucket}/sdk/.env: {exc}")

    whl_filename = "foldrun_shim-0.1.0-py3-none-any.whl"
    if bucket_configured:
        try:
            latest_build_dst = Path("/tmp/foldrun_latest_build")
            res = subprocess.run(
                [
                    "gcloud",
                    "storage",
                    "cp",
                    f"gs://{clean_bucket}/sdk/latest_build",
                    str(latest_build_dst),
                ],
                check=False,
                capture_output=True,
                text=True,
            )
            if res.returncode == 0 and latest_build_dst.is_file():
                candidate = latest_build_dst.read_text(encoding="utf-8").strip()
                if candidate.endswith(".whl"):
                    whl_filename = candidate
                    print(
                        f"Detected latest FoldRun SDK build from gs://{clean_bucket}/sdk/latest_build: {whl_filename}"
                    )
        except Exception as exc:
            print(f"Notice: could not read gs://{clean_bucket}/sdk/latest_build: {exc}")

    whl_path = Path(f"/tmp/{whl_filename}")
    marker_path = Path(f"/tmp/{whl_filename}.installed")
    if marker_path.exists():
        try:
            import foldrun_shim

            _ensure_scripts_on_path()
            return
        except ImportError:
            pass

    if not bucket_configured:
        try:
            import foldrun_shim

            _ensure_scripts_on_path()
            print(
                "Notice: FOLDRUN_GCS_BUCKET is unconfigured; using locally installed foldrun_shim."
            )
            return
        except ImportError:
            raise RuntimeError(
                f"foldrun_shim is not installed. Set FOLDRUN_GCS_BUCKET='<PROJECT_ID>-foldrun-data' "
                f"so this cell can download gs://<PROJECT_ID>-foldrun-data/sdk/{whl_filename}."
            )

    gcs_uri = f"gs://{clean_bucket}/sdk/{whl_filename}"
    print(f"Downloading {whl_filename} from {gcs_uri}...")
    cp_res = subprocess.run(
        ["gcloud", "storage", "cp", gcs_uri, str(whl_path)],
        check=False,
        capture_output=True,
        text=True,
    )
    if cp_res.returncode != 0:
        err_line = (
            (cp_res.stderr or cp_res.stdout or "gcloud storage cp failed").strip().splitlines()[-1]
        )
        try:
            import foldrun_shim  # noqa: F401

            _ensure_scripts_on_path()
            print(
                f"Warning: could not download {gcs_uri} ({err_line}); using locally installed foldrun_shim."
            )
            return
        except ImportError:
            raise RuntimeError(
                f"Failed to download {gcs_uri} ({err_line}) and foldrun_shim is not installed locally."
            )

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", str(whl_path)],
        check=True,
    )
    _ensure_scripts_on_path()
    marker_path.write_text("installed", encoding="utf-8")
    print(f"Installed {whl_filename} successfully.")


_install_foldrun_shim_from_gcs(os.environ["FOLDRUN_GCS_BUCKET"])
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "py3Dmol",
        "matplotlib",
        "numpy",
        "pyyaml",
    ],
    check=False,
)

import yaml

import foldrun_shim
from foldrun_shim import boltz
from foldrun_shim.bundler import bundle_local_inputs
from foldrun_shim.cli import _strip_local_boltz_args

print(f"foldrun-shim version : {getattr(foldrun_shim, '__version__', 'unknown')}")
print(f"boltz CLI in PATH    : {shutil.which('boltz') or 'available via foldrun_shim.cli'}")
print(f"FOLDRUN_GATEWAY_URL  : {os.environ['FOLDRUN_GATEWAY_URL']}")

In [ ]:
# @title 1B. (Optional) Check Live GPU Availability & Configure On-Demand vs. DWS Scheduling
# @markdown Thin-permission notebook users (who only have `roles/run.invoker` on the FoldRun Gateway)
# @markdown can inspect live GPU availability, active/queued jobs, and regional quotas directly from Python
# @markdown without needing GCP project-level GCE or ServiceUsage IAM permissions.
# @markdown
# @markdown **CLI Equivalents:**
# @markdown - `!foldrun gpus` (inspect live GPU availability and queue depths)
# @markdown - `!foldrun status <job_id>` (inspect status of an active or queued job)
# @markdown - `!foldrun cancel <job_id>` (cancel a queued or running job)
# @markdown - GCP Project Admins only: `!gcloud alpha services quota list --service=aiplatform.googleapis.com --consumer=projects/$GCP_PROJECT_ID`

import os

try:
    from IPython.display import display
except ImportError:
    display = print

# 1. Query live GPU availability and queue status via FoldRun Gateway
try:
    import foldrun_shim

    gpu_status = foldrun_shim.check_gpus()
    display(gpu_status)
except Exception as exc:
    print(f"Notice: Could not query live GPU status via FoldRun Gateway: {exc}")
    print(
        "You can still proceed with prediction using default scheduling or environment variables below."
    )

# 2. Scheduling & Hardware Configuration (Optional Overrides)
# Uncomment or adjust any of the following environment variables to control hardware & scheduling:

# GPU Architecture: "auto" (default), "L4" (fast & cost-effective), "A100" (40GB), or "A100_80GB"
# os.environ["FOLDRUN_GPU_TYPE"] = "L4"

# Scheduling Strategy:
# - "true"  -> Dynamic Workload Scheduler (DWS) FLEX_START (saves 45-68% cost; may queue if regional DWS capacity is constrained)
# - "false" -> Immediate On-Demand STANDARD GPUs (provisions immediately without DWS queueing)
# os.environ["FOLDRUN_ENABLE_FLEX_START"] = "false"

# Optional Queue Timeout: Automatically cancel the job if queued in DWS longer than N seconds
# os.environ["FOLDRUN_QUEUE_TIMEOUT_SEC"] = "300"

---
## Part A — Original Upstream Example (Unmodified / Verbatim Workflow)

### Original `jwohlwend/boltz` Upstream Examples

In the official [`jwohlwend/boltz` repository](https://github.com/jwohlwend/boltz), inference input files are authored as simple YAML documents specifying target sequences, ligands (via SMILES or CCD codes), and optional binding affinity properties (`examples/prot.yaml`, `examples/ligand.yaml`, and `examples/affinity.yaml`).

We author all three canonical upstream examples and run the exact upstream command:
```bash
!boltz predict prot.yaml --use_msa_server --out_dir ./boltz_out
```


In [ ]:
# Write exact upstream jwohlwend/boltz examples/prot.yaml, examples/ligand.yaml, and examples/affinity.yaml
from pathlib import Path

prot_yaml = """version: 1
sequences:
  - protein:
      id: A
      sequence: MVTPEGNLVRVAVQALAEKLESEKVKVVDMEVKELEKTVDKLKEELEA
"""
Path("prot.yaml").write_text(prot_yaml, encoding="utf-8")

# Ligand co-folding (examples/ligand.yaml with CCD: SAH)
ligand_yaml = """version: 1
sequences:
  - protein:
      id: A
      sequence: MVTPEGNLVRVAVQALAEKLESEKVKVVDMEVKELEKTVDKLKEELEA
  - ligand:
      id: B
      ccd: SAH
"""
Path("ligand.yaml").write_text(ligand_yaml, encoding="utf-8")

# Affinity prediction (examples/affinity.yaml)
affinity_yaml = """version: 1
sequences:
  - protein:
      id: A
      sequence: MVTPEGNLVRVAVQALAEKLESEKVKVVDMEVKELEKTVDKLKEELEA
  - ligand:
      id: B
      ccd: SAH
properties:
  - affinity:
      binder: B
"""
Path("affinity.yaml").write_text(affinity_yaml, encoding="utf-8")
print("Wrote prot.yaml, ligand.yaml, and affinity.yaml")

In [ ]:
# Verbatim upstream Boltz-2 command execution:
# In upstream Boltz, --use_msa_server queries the public MSA server.
# foldrun-shim intercepts --use_msa_server so sequences stay strictly private and align against /mnt/nfs/foldrun.
!boltz predict prot.yaml --use_msa_server --out_dir ./boltz_out/prot

---
## Part B — FoldRun Additions (`foldrun-shim` Python API, `/mnt/nfs/foldrun` Shared Databases & Weights, Local Asset Bundling, and Multi-GPU Seed Parallelism)

FoldRun enhances Boltz-2 for institutional drug discovery:
1. **Private Server-Side MSA Search**: Intercepts `--use_msa_server` so proprietary target sequences never leave your VPC, aligning against 2.5 TB `/mnt/nfs/foldrun` (`uniref90`, `mgnify`) with SHA-256 caching (`/mnt/nfs/foldrun/boltz2_msas_cache/`).
2. **Local Pre-computed MSA & Template Bundling**: Transparently bundles local `.a3m` and `.cif` files into `assets.tar.gz` and rewrites YAML paths to GCS FUSE locations.
3. **Multi-Seed GPU Fan-Out**: Runs up to 50 parallel seeds (`--num_model_seeds=5`) across Vertex AI A100 GPUs with DWS `FLEX_START`.
4. **Python Notebook Helper API (`boltz.predict`)**: Clean programmatic interface with status cards and automatic download of predictions and PAE/PDE matrices.


## 2. How FoldRun's Cloud Filestore NFS Database Is Leveraged (3 Alignment Modes)

Every Boltz-2 pipeline in FoldRun mounts a shared **2.5 TB `BASIC_SSD` Cloud Filestore NFS volume** at `/mnt/nfs/foldrun` (`foldrun-agent/foldrun_app/databases.yaml`). Depending on how you write your Boltz-2 `.yaml` input, FoldRun supports **three distinct database and alignment modes**:

### Mode A — Server-Side Cloud Filestore NFS Database Search (Default)
- **How to trigger**: Omit the `msa:` field on your `protein` entry in the Boltz-2 YAML (or leave `--use_msa_server` on the CLI, which `foldrun-shim` automatically intercepts and ignores).
- **What happens on FoldRun**:
  1. `msa_pipeline_boltz2` runs on a `c2-standard-16` CPU VM with `/mnt/nfs/foldrun` mounted.
  2. For each protein sequence, it computes `seq_hash = sha256(sequence)` and checks `/mnt/nfs/foldrun/boltz2_msas_cache/protein_<seq_hash>/combined.a3m`.
  3. **On a cache miss**, it executes `jackhmmer --noali -N 1 --cpu 8` against `/mnt/nfs/foldrun/uniref90/uniref90.fasta` and `/mnt/nfs/foldrun/mgnify/mgy_clusters_2022_05.fa`, converts Stockholm (`.sto`) to `.a3m`, concatenates the alignments, strips intermediate files, and atomically promotes the directory to `/mnt/nfs/foldrun/boltz2_msas_cache/protein_<seq_hash>/combined.a3m`.
  4. **On a cache hit**, `jackhmmer` is skipped entirely and the cached `.a3m` path is injected into the query YAML.
  5. `predict_boltz2` runs on parallel NVIDIA A100 GPUs with `--cache=/mnt/nfs/foldrun/boltz2/cache`, loading pre-staged `boltz2_conf.ckpt`, `boltz2_aff.ckpt`, and the Chemical Component Dictionary (`mols/`).

### Mode B — Bundling Local Pre-Computed Custom MSAs (`.a3m`) & Structural Templates (`.cif` / `.pdb`)
- **How to trigger**: Reference relative local paths in your Boltz-2 YAML (`sequences[*].protein.msa: ./custom_alignment.a3m` and/or `templates: [{cif: ./template.cif}]`). Note: Ensure your YAML begins with `version: 1`.
- **What happens on FoldRun**:
  1. `foldrun_shim.bundler._collect_boltz_assets` parses the YAML, verifies path containment inside the input directory, and packages all referenced `.a3m`, `.cif`, and `.pdb` files into an in-memory `assets.tar.gz`.
  2. `foldrun-gateway` validates and extracts `assets.tar.gz` to `gs://<bucket>/gateway_jobs/<job_id>/inputs/assets/` and rewrites the relative paths in the staged YAML to `/gcs/<bucket>/gateway_jobs/<job_id>/inputs/assets/...`.
  3. `msa_pipeline_boltz2` detects that `prot_data['msa']` already points to an existing `/gcs/...` file, **preserves your custom MSA, and skips `jackhmmer`**, while `predict_boltz2` still reads `boltz2_conf.ckpt`, `boltz2_aff.ckpt`, and `mols/` from `/mnt/nfs/foldrun/boltz2/cache`.

### Mode C — MSA-Free / Single-Sequence Mode (`msa: empty`)
- **How to trigger**: Set `msa: empty` on a `protein` chain (useful for *de novo* designed binders or synthetic scaffolds with no natural homologs).
- **What happens on FoldRun**:
  1. `foldrun-shim` recognizes the special keyword `empty` and uploads only the YAML file.
  2. `msa_pipeline_boltz2` preserves `msa: empty` and **skips `jackhmmer`** for that chain, proceeding directly to A100 GPU structure prediction.

--- 
## 3. Mode A (Interface 1): Drop-In Native CLI (`!boltz predict ...`) with Server-Side NFS MSA & Binding Affinity

First, we adapt the official [`examples/affinity.yaml`](https://github.com/jwohlwend/boltz/blob/main/examples/affinity.yaml) from `jwohlwend/boltz`, which folds a 388-residue enzyme receptor (Chain `A`) in complex with a small-molecule ligand (`N[C@@H](Cc1ccc(O)cc1)C(=O)O`, Chain `B`) and enables quantitative binding affinity prediction via `properties: [{affinity: {binder: B}}]`.

Notice that:
- We omit `msa:` on Chain `A` so FoldRun executes **Mode A** (`jackhmmer` against `/mnt/nfs/foldrun/uniref90` + `/mnt/nfs/foldrun/mgnify` with SHA-256 caching in `/mnt/nfs/foldrun/boltz2_msas_cache/`).
- In the CLI command below, even if a public notebook includes `--use_msa_server`, `--cache`, `--devices`, or `--accelerator`, `foldrun-shim` automatically intercepts and strips those local flags while forwarding allowlisted scientific flags (`--use_potentials`, `--diffusion_samples 5`, `--sampling_steps_affinity 200`) to `foldrun-gateway`.

In [ ]:
# Official jwohlwend/boltz examples/affinity.yaml (Mode A: no local msa path -> Server-Side NFS Search)
affinity_yaml_content = """version: 1
sequences:
  - protein:
      id: A
      sequence: MVTPEGNVSLVDESLLVGVTDEDRAVRSAHQFYERLIGLWAPAVMEAAHELGVFAALAEAPADSGELARRLDCDARAMRVLLDALYAYDVIDRIHDTNGFRYLLSAEARECLLPGTLFSLVGKFMHDINVAWPAWRNLAEVVRHGARDTSGAESPNGIAQEDYESLVGGINFWAPPIVTTLSRKLRASGRSGDATASVLDVGCGTGLYSQLLLREFPRWTATGLDVERIATLANAQALRLGVEERFATRAGDFWRGGWGTGYDLVLFANIFHLQTPASAVRLMRHAAACLAPDGLVAVVDQIVDADREPKTPQDRFALLFAASMTNTGGGDAYTFQEYEEWFTAAGLQRIETLDTPMHRILLARRATEPSAVPEGQASENLYFQ
  - ligand:
      id: B
      smiles: 'N[C@@H](Cc1ccc(O)cc1)C(=O)O'
properties:
  - affinity:
      binder: B
"""

Path("affinity.yaml").write_text(affinity_yaml_content, encoding="utf-8")

# Show the deterministic NFS MSA cache path that FoldRun's msa_pipeline_boltz2 computes for Chain A
parsed_affinity = yaml.safe_load(affinity_yaml_content)
chain_a_seq = parsed_affinity["sequences"][0]["protein"]["sequence"]
chain_a_sha256 = hashlib.sha256(chain_a_seq.encode()).hexdigest()
print(f"Wrote affinity.yaml ({len(chain_a_seq)} aa protein + SMILES ligand)")
print(
    f"FoldRun NFS MSA cache path: /mnt/nfs/foldrun/boltz2_msas_cache/protein_{chain_a_sha256}/combined.a3m"
)

# Verify how foldrun-shim strips local Colab/host flags while preserving allowlisted scientific flags
sample_cli = [
    "predict",
    "affinity.yaml",
    "--use_msa_server",
    "--cache",
    "~/.boltz",
    "--devices",
    "1",
    "--accelerator",
    "gpu",
    "--use_potentials",
    "--diffusion_samples",
    "5",
    "--sampling_steps_affinity",
    "200",
    "--out_dir",
    "./boltz_out",
]
data_path, out_dir, override, scientific_args = _strip_local_boltz_args(sample_cli)
print(f"Sanitized scientific CLI args forwarded to foldrun-gateway: {scientific_args}")

In [ ]:
# Interface 1: Drop-In Native Boltz-2 CLI execution via foldrun-shim
# Uses a dedicated output subdirectory (./boltz_out/affinity_cli) so .foldrun_job.json never collides across examples
!boltz predict affinity.yaml --use_potentials --diffusion_samples 5 --sampling_steps_affinity 200 --out_dir ./boltz_out/affinity_cli

---
## 4. Mode A (Interface 2): Python Notebook Helper API (`boltz.predict`) & Pocket-Conditioned Docking

In addition to the `!boltz predict` CLI wrapper, `foldrun-shim` provides a native Python Notebook Helper (`from foldrun_shim import boltz`) that:
1. Accepts allowlisted Boltz-2 scientific flags as Python keyword arguments:
   - **Pipeline scaling flags**: `diffusion_samples` (`1..25`), `num_model_seeds` (`1..10`), `seed` (`int`).
   - **Container diffusion & MSA flags**: `recycling_steps` (`1..20`), `sampling_steps` (`1..1000`), `step_scale` (`0.1..10.0`), `use_potentials` (`bool`), `write_full_pae` (`bool`), `write_full_pde` (`bool`), `max_msa_seqs` (`32..16384`), `subsample_msa` (`bool`), `num_subsampled_msa` (`1..4096`).
   - **Binding affinity flags**: `sampling_steps_affinity` (`1..1000`), `diffusion_samples_affinity` (`1..25`), `affinity_mw_correction` (`bool`).
2. Renders an interactive HTML status card in Jupyter (`GatewayJobResult._repr_html_()`) with a direct link to the 3D FoldRun Viewer (`job.viewer_url`).
3. Materializes the multi-seed predictions into `<out_dir>/boltz_results_<stem>/predictions/<stem>/`.

Below we also write [`pocket.yaml`](https://github.com/jwohlwend/boltz/blob/main/examples/pocket.yaml) (adapted from `jwohlwend/boltz/examples/pocket.yaml`), which uses a Chemical Component Dictionary ligand (`ccd: EKY`, resolved on FoldRun from `/mnt/nfs/foldrun/boltz2/cache/mols/`) and spatial `pocket` contact constraints.

In [ ]:
# Interface 2: Python Notebook Helper API with multi-seed A100 fan-out (2 seeds x 5 samples = 10 structures),
# inference-time potentials, full PAE/PDE matrix output, and MW-corrected binding affinity
job = boltz.predict(
    "affinity.yaml",
    out_dir="./boltz_out/affinity",
    use_potentials=True,
    diffusion_samples=5,
    num_model_seeds=2,
    seed=42,
    recycling_steps=3,
    sampling_steps=200,
    step_scale=1.5,
    write_full_pae=True,
    write_full_pde=True,
    sampling_steps_affinity=200,
    diffusion_samples_affinity=5,
    affinity_mw_correction=True,
)

# Display rich HTML summary card with FoldRun 3D Viewer link
job

In [ ]:
# Official jwohlwend/boltz examples/pocket.yaml adapted with 'version: 1'
# Demonstrates CCD ligand lookup ('ccd: EKY') from /mnt/nfs/foldrun/boltz2/cache/mols/ + pocket constraints
pocket_yaml_content = """version: 1
sequences:
  - protein:
      id: [A1]
      sequence: MYNMRRLSLSPTFSMGFHLLVTVSLLFSHVDHVIAETEMEGEGNETGECTGSYYCKKGVILPIWEPQDPSFGDKIARATVYFVAMVYMFLGVSIIADRFMSSIEVITSQEKEITIKKPNGETTKTTVRIWNETVSNLTLMALGSSAPEILLSVIEVCGHNFTAGDLGPSTIVGSAAFNMFIIIALCVYVVPDGETRKIKHLRVFFVTAAWSIFAYTWLYIILSVISPGVVEVWEGLLTFFFFPICVVFAWVADRRLLFYKYVYKRYRAGKQRGMIIEHEGDRPSSKTEIEMDGKVVNSHVENFLDGALVLEVDERDQDDEEARREMARILKELKQKHPDKEIEQLIELANYQVLSQQQKSRAFYRIQATRLMTGAGNILKRHAADQARKAVSMHEVNTEVTENDPVSKIFFEQGTYQCLENCGTVALTIIRRGGDLTNTVFVDFRTEDGTANAGSDYEFTEGTVVFKPGDTQKEIRVGIIDDDIFEEDENFLVHLSNVKVSSEASEDGILEANHVSTLACLGSPSTATVTIFDDDHAGIFTFEEPVTHVSESIGIMEVKVLRTSGARGNVIVPYKTIEGTARGGGEDFEDTCGELEFQNDEIVKIITIRIFDREEYEKECSFSLVLEEPKWIRRGMKGGFTITDEYDDKQPLTSKEEEERRIAEMGRPILGEHTKLEVIIEESYEFKSTVDKLIKKTNLALVVGTNSWREQFIEAITVSAGEDDDDDECGEEKLPSCFDYVMHFLTVFWKVLFAFVPPTEYWNGWACFIVSILMIGLLTAFIGDLASHFGCTIGLKDSVTAVVFVALGTSVPDTFASKVAATQDQYADASIGNVTGSNAVNVFLGIGVAWSIAAIYHAANGEQFKVSPGTLAFSVTLFTIFAFINVGVLLYRRRPEIGGELGGPRTAKLLTSCLFVLLWLLYIFFSSLEAYCHIKGF
  - ligand:
      ccd: EKY
      id: [B1]
constraints:
  - pocket:
      binder: B1
      contacts: [[A1, 829], [A1, 138]]
      max_distance: 6.0
"""

Path("pocket.yaml").write_text(pocket_yaml_content, encoding="utf-8")
print(
    "Wrote pocket.yaml (940 aa receptor + CCD ligand EKY + pocket constraint on residues 829 & 138)"
)

# Optional: Submit pocket-conditioned prediction into its own output subdirectory
# pocket_job = boltz.predict("pocket.yaml", out_dir="./boltz_out/pocket", use_potentials=True, diffusion_samples=5)

In [ ]:
# Inspect <out_dir>/.foldrun_job.json written by foldrun-shim for interrupt resilience
candidate_state_files = [
    Path("./boltz_out/affinity/.foldrun_job.json"),
    Path("./boltz_out/affinity_cli/.foldrun_job.json"),
    Path("./boltz_out/prot/.foldrun_job.json"),
]
state_file = next((p for p in candidate_state_files if p.exists()), candidate_state_files[0])
if state_file.exists():
    state_data = json.loads(state_file.read_text(encoding="utf-8"))
    print(f"Saved interrupt-recovery state ({state_file}):")
    print(json.dumps(state_data, indent=2))
else:
    print(
        f"No .foldrun_job.json found yet at {state_file} (run the prediction cell above against a live gateway)."
    )

---
## 5. Mode B: Bundling Local Pre-Computed Custom MSAs (`.a3m`) & Structural Templates (`.cif`)

Upstream `jwohlwend/boltz` supports supplying custom alignments (`examples/prot_custom_msa.yaml`) and structural templates (`templates: [{cif: ...}]`). In a cloud environment, local relative paths normally fail because remote GPU workers cannot see your laptop or Colab filesystem.

`foldrun-shim` solves this automatically:
1. `foldrun_shim.bundler._collect_boltz_assets` parses any Boltz-2 YAML starting with `version:` and collects every file referenced in `sequences[*].protein.msa` (unless set to `empty`) and `templates[*].cif` / `templates[*].pdb`.
2. All referenced local files are validated for path containment (no `..` or leading `/`) and compressed into `assets.tar.gz`.
3. `foldrun-gateway` stages the files to `gs://<bucket>/gateway_jobs/<job_id>/inputs/assets/` and rewrites the YAML paths to `/gcs/<bucket>/gateway_jobs/<job_id>/inputs/assets/...`.
4. On the Vertex AI worker, `msa_pipeline_boltz2` sees that `prot_data['msa']` already exists on `/gcs/...`, **skips `jackhmmer`**, and passes your custom `.a3m` and `.cif` template directly to `predict_boltz2` (which still uses `/mnt/nfs/foldrun/boltz2/cache` for model checkpoints and CCD ligands).

In [ ]:
# 1. Create a local pre-computed custom MSA (.a3m) adapted from jwohlwend/boltz/examples/prot_custom_msa.yaml
custom_a3m_content = """>query_protein
QLEDSEVEAVAKGLEEMYANGVTEDNFKNYVKNNFAQQEISSVEEELNVNISDSCVANKIKDEFFAMISISAIVKAAQKKAWKELAVTVLRFAKANGLKTNAIIVAGQLALWAVQCG
>homolog_1
QLEDSEVEAVAKGLEEMYANGVTEDNFKNYVKNNFAQQEISSVEEELNVNISDSCVANKIKDEFFAMISISAIVKAAQKKAWKELAVTVLRFAKANGLKTNAIIVAGQLALWAVQCG
>homolog_2
QLEDSEVEAVAKGLEEMYANGVTEDNFRNYVKNNFAQQEISSVEEELNVNISDSCVANKIKDEFFAMISISAIVKAAQKKAWKELAVTVLRFAKANGLKTNAIIVAGQLALWAVQCG
"""
Path("custom_alignment.a3m").write_text(custom_a3m_content, encoding="utf-8")

# 2. Create a local structural template (.cif)
template_cif_content = """data_template
#
_entry.id template
loop_
_atom_site.group_PDB
_atom_site.id
_atom_site.type_symbol
_atom_site.label_atom_id
_atom_site.label_comp_id
_atom_site.label_asym_id
_atom_site.label_seq_id
_atom_site.Cartn_x
_atom_site.Cartn_y
_atom_site.Cartn_z
_atom_site.occupancy
_atom_site.B_iso_or_equiv
_atom_site.auth_asym_id
ATOM 1 N N GLN A 1 10.120 12.450 8.310 1.00 85.00 A
ATOM 2 C CA GLN A 1 11.420 12.980 8.720 1.00 85.00 A
ATOM 3 C C GLN A 1 12.510 11.920 8.610 1.00 85.00 A
ATOM 4 O O GLN A 1 12.310 10.740 8.320 1.00 85.00 A
#
"""
Path("template.cif").write_text(template_cif_content, encoding="utf-8")

# 3. Write custom_complex.yaml referencing both ./custom_alignment.a3m and ./template.cif
custom_complex_yaml = """version: 1
sequences:
  - protein:
      id: A
      sequence: QLEDSEVEAVAKGLEEMYANGVTEDNFKNYVKNNFAQQEISSVEEELNVNISDSCVANKIKDEFFAMISISAIVKAAQKKAWKELAVTVLRFAKANGLKTNAIIVAGQLALWAVQCG
      msa: ./custom_alignment.a3m
  - ligand:
      id: B
      ccd: SAH
templates:
  - cif: ./template.cif
    chain_id: A
    force: true
    threshold: 2.0
"""
Path("custom_complex.yaml").write_text(custom_complex_yaml, encoding="utf-8")

# Inspect the in-memory bundle created by foldrun_shim.bundler
bundled_mode_b = bundle_local_inputs(
    model="boltz",
    input_path="custom_complex.yaml",
    cli_args=["--use_potentials", "--diffusion_samples", "5"],
)
print(f"Bundled assets in assets.tar.gz : {list(bundled_mode_b.assets_map.keys())}")
print(f"Archive size (compressed bytes) : {len(bundled_mode_b.assets_tar_bytes or b'')}")
print(f"Deterministic client_input_sha256: {bundled_mode_b.client_input_sha256}")

# Submit Mode B prediction via boltz.predict into dedicated ./boltz_out/custom_msa directory
custom_job = boltz.predict(
    "custom_complex.yaml",
    out_dir="./boltz_out/custom_msa",
    use_potentials=True,
    diffusion_samples=5,
)
custom_job

---
## 6. Mode C: MSA-Free / Single-Sequence Mode (`msa: empty`) for *De Novo* Designed Proteins

When evaluating *de novo* designed proteins, cyclic peptides (`examples/cyclic_prot.yaml`), or synthetic scaffolds (`examples/prot_no_msa.yaml`) where no natural evolutionary homologs exist in UniRef90 or MGnify, set `msa: empty` on the protein chain.

- `foldrun_shim.bundler` recognizes `msa: empty` as a reserved keyword rather than a local file.
- `msa_pipeline_boltz2` preserves `msa: empty` and skips `jackhmmer` completely.

In [ ]:
# Adapted from jwohlwend/boltz/examples/prot_no_msa.yaml (Mode C: msa: empty)
de_novo_yaml = """version: 1
sequences:
  - protein:
      id: A
      sequence: QLEDSEVEAVAKGLEEMYANGVTEDNFKNYVKNNFAQQEISSVEEELNVNISDSCVANKIKDEFFAMISISAIVKAAQKKAWKELAVTVLRFAKANGLKTNAIIVAGQLALWAVQCG
      msa: empty
"""
Path("de_novo_binder.yaml").write_text(de_novo_yaml, encoding="utf-8")

bundled_mode_c = bundle_local_inputs(
    model="boltz",
    input_path="de_novo_binder.yaml",
    cli_args=["--diffusion_samples", "5", "--recycling_steps", "5"],
)
print(f"Bundled assets for msa: empty : {list(bundled_mode_c.assets_map.keys())} (expected: [])")

# Run single-sequence prediction on A100 GPU into dedicated ./boltz_out/de_novo directory
denovo_job = boltz.predict(
    "de_novo_binder.yaml",
    out_dir="./boltz_out/de_novo",
    diffusion_samples=5,
    recycling_steps=5,
)
denovo_job

---
## 7. Downstream Analysis & Visualization of Materialized Native Outputs

When the Vertex AI pipeline completes, `foldrun-gateway` (`output_packager.py`) streams all per-seed artifacts from GCS into `native_outputs.tar.gz`, re-indexing samples across seeds (`0 .. num_model_seeds * diffusion_samples - 1`) and reconstructing the exact native Boltz-2 directory hierarchy inside `--out_dir`:

```text
boltz_out/
├── .foldrun_job.json
└── boltz_results_affinity/
    └── predictions/
        └── affinity/
            ├── affinity_model_0.cif
            ├── confidence_affinity_model_0.json
            ├── pae_affinity_model_0.npz
            ├── pde_affinity_model_0.npz
            ├── ...
            └── affinity_affinity.json
```

Because this matches local `boltz predict` output paths byte-for-byte, standard downstream analysis cells (`json`, `numpy`, `matplotlib`, `py3Dmol`) work without modification.

In [ ]:
# 7.1 Inspect Confidence & Quantitative Binding Affinity JSON Outputs
candidate_pred_dirs = [
    Path("./boltz_out/affinity/boltz_results_affinity/predictions/affinity"),
    Path("./boltz_out/affinity_cli/boltz_results_affinity/predictions/affinity"),
]
pred_dir = next((p for p in candidate_pred_dirs if p.exists()), candidate_pred_dirs[0])

if pred_dir.exists():
    conf_files = sorted(pred_dir.glob("confidence_affinity_model_*.json"))
    print(f"Found {len(conf_files)} model confidence files in {pred_dir}:\n")
    print(
        f"{'Model':<22} {'Conf Score':>10} {'pTM':>8} {'ipTM':>8} {'Lig ipTM':>10} {'Complex pLDDT':>14} {'Complex iPDE (A)':>17}"
    )
    print("-" * 95)
    for cf in conf_files:
        c = json.loads(cf.read_text(encoding="utf-8"))
        model_label = cf.stem.replace("confidence_", "")
        print(
            f"{model_label:<22} "
            f"{c.get('confidence_score', 0.0):>10.4f} "
            f"{c.get('ptm', 0.0):>8.4f} "
            f"{c.get('iptm', 0.0):>8.4f} "
            f"{c.get('ligand_iptm', 0.0):>10.4f} "
            f"{c.get('complex_plddt', 0.0):>14.4f} "
            f"{c.get('complex_ipde', 0.0):>17.4f}"
        )

    affinity_file = pred_dir / "affinity_affinity.json"
    if affinity_file.exists():
        aff = json.loads(affinity_file.read_text(encoding="utf-8"))
        log10_ic50_um = float(aff.get("affinity_pred_value", 0.0))
        binder_prob = float(aff.get("affinity_probability_binary", 0.0))
        ic50_um = 10.0**log10_ic50_um
        ic50_nm = ic50_um * 1000.0
        pic50 = 6.0 - log10_ic50_um
        delta_g_kcal = (6.0 - log10_ic50_um) * 1.364

        print("\n=== Boltz-2 Quantitative Binding Affinity (affinity_affinity.json) ===")
        print(f"  affinity_probability_binary (hit probability) : {binder_prob:.4f}")
        print(f"  affinity_pred_value (log10 IC50 in uM)        : {log10_ic50_um:.4f}")
        print(
            f"  Estimated IC50                                : {ic50_um:.4f} uM ({ic50_nm:.2f} nM)"
        )
        print(f"  Estimated pIC50                               : {pic50:.2f}")
        print(f"  Estimated Binding Free Energy (dG)            : {delta_g_kcal:.2f} kcal/mol")
else:
    print(f"Prediction directory {pred_dir} not materialized yet.")

In [ ]:
# 7.2 Plot PAE & PDE Error Matrices (pae_affinity_model_0.npz & pde_affinity_model_0.npz)
import matplotlib.pyplot as plt
import numpy as np

pae_path = pred_dir / "pae_affinity_model_0.npz"
pde_path = pred_dir / "pde_affinity_model_0.npz"

if pae_path.exists() or pde_path.exists():
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    if pae_path.exists():
        pae_npz = np.load(pae_path)
        pae_matrix = pae_npz[pae_npz.files[0]]
        im0 = axes[0].imshow(pae_matrix, cmap="Greens_r", vmin=0.0, vmax=31.75)
        axes[0].set_title("Predicted Aligned Error (PAE) - Model 0")
        axes[0].set_xlabel("Scored Token / Residue")
        axes[0].set_ylabel("Aligned Token / Residue")
        plt.colorbar(im0, ax=axes[0], label="Expected Position Error (Å)")

    if pde_path.exists():
        pde_npz = np.load(pde_path)
        pde_matrix = pde_npz[pde_npz.files[0]]
        im1 = axes[1].imshow(pde_matrix, cmap="viridis_r", vmin=0.0, vmax=16.0)
        axes[1].set_title("Predicted Distance Error (PDE) - Model 0")
        axes[1].set_xlabel("Token Index")
        axes[1].set_ylabel("Token Index")
        plt.colorbar(im1, ax=axes[1], label="Distance Error (Å)")

    plt.tight_layout()
    plt.show()
else:
    print(
        "PAE/PDE .npz files not found yet; run boltz.predict(..., write_full_pae=True, write_full_pde=True) first."
    )

In [ ]:
# 7.3 Interactive 3D Visualization of Protein-Ligand Complex (affinity_model_0.cif) with py3Dmol
import py3Dmol

cif_path = pred_dir / "affinity_model_0.cif"

if cif_path.exists():
    cif_data = cif_path.read_text(encoding="utf-8")
    view = py3Dmol.view(width=820, height=540)
    view.addModel(cif_data, "cif")

    # Style protein receptor (Chain A) colored by per-token pLDDT stored in B-factor column
    view.setStyle(
        {"chain": "A"},
        {
            "cartoon": {
                "colorscheme": {
                    "prop": "b",
                    "gradient": "roygb",
                    "min": 50,
                    "max": 90,
                }
            }
        },
    )
    # Highlight small-molecule ligand binder (Chain B) as sticks + translucent surface
    view.setStyle({"chain": "B"}, {"stick": {"colorscheme": "cyanCarbon", "radius": 0.25}})
    view.addSurface(py3Dmol.VDW, {"opacity": 0.65, "color": "cyan"}, {"chain": "B"})

    view.zoomTo()
    view.show()
else:
    print(f"Structure file {cif_path} not materialized yet.")

if "job" in globals() and getattr(job, "viewer_url", None):
    print(f"Full interactive FoldRun 3D Viewer URL: {job.viewer_url}")